# ⚡ Fantasy World Predictions — Harry Potter Universe
## Step 2: Cleaning

---

### 🎯 Goal of this notebook
Fix every issue flagged in the EDA notebook and produce a clean dataframe ready for modeling.

**Decisions:**
- `Gender`: drop rows still missing after checking (no label variants to normalize here, unlike LOTR)
- `Species/Race`: normalize casing, merge the ambiguous `"witch or wizard"` value, group rare
  classes into `Other`, **drop** rows still labeled `unknown`
- `Blood`: normalize casing only — **keep** `Unknown` as its own class (weak target, dropping
  80% of rows would gut the dataset for no benefit)
- `School` → split into `is_hogwarts` (binary) and `hogwarts_house` (categorical, `Unknown`/`None`
  kept as categories)
- `Profession` → grouped into broader `profession_group` buckets, `Unknown` kept as a category
- `Name` kept (display only, not a feature). `Link` dropped. `Descr` → engineered into
  `has_description` binary, then dropped as raw text

---

## 📦 Section 1: Imports & Load

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/HPCharactersData.csv')
print(f'Starting shape: {df.shape}')

## 🚻 Section 2: Clean `Gender`

Already clean labels (`Male`/`Female`) — just drop rows where it's missing.

In [ ]:
before = len(df)
df = df[df['Gender'].isin(['Male', 'Female'])].copy()
print(f'Dropped {before - len(df)} rows with missing gender. New shape: {df.shape}')
df['Gender'].value_counts()

## 🧬 Section 3: Clean `Species/Race`

Normalize casing, merge the ambiguous `"witch or wizard"` label into `wizard`, group rare
classes (fewer than 5 occurrences) into `Other`, then drop rows still labeled `unknown`.

In [ ]:
df['Species/Race'] = df['Species/Race'].astype(str).str.strip().str.lower()
df['Species/Race'] = df['Species/Race'].replace({'witch or wizard': 'wizard'})

before = len(df)
df = df[df['Species/Race'] != 'unknown'].copy()
print(f'Dropped {before - len(df)} rows labeled unknown species/race.')

race_counts = df['Species/Race'].value_counts()
rare_races = race_counts[race_counts < 5].index
df['Species/Race'] = df['Species/Race'].apply(lambda x: 'other' if x in rare_races else x)

print(f'New shape: {df.shape}')
df['Species/Race'].value_counts()

## 🩸 Section 4: Clean `Blood`

Normalize casing only. `Unknown` stays as its own class — dropping it would remove ~80% of
rows for a column we already know is a weak target.

In [ ]:
df['Blood'] = df['Blood'].astype(str).str.strip().str.title()
df['Blood'].value_counts()

## 🏫 Section 5: Engineer `is_hogwarts` and `hogwarts_house` from `School`

`School` bundles school + house/role into one string, sometimes two joined with `\\`. Split
into: whether the character is Hogwarts-affiliated at all, and which house (if any/known).

In [ ]:
houses = ['Gryffindor', 'Slytherin', 'Ravenclaw', 'Hufflepuff']

def is_hogwarts(val):
    return 1 if 'Hogwarts' in str(val) else 0

def extract_house(val):
    s = str(val)
    for h in houses:
        if h in s:
            return h
    return 'None'  # Hogwarts staff/student w/o a listed house, or non-Hogwarts, or literal Unknown

df['is_hogwarts'] = df['School'].apply(is_hogwarts)
df['hogwarts_house'] = df['School'].apply(extract_house)

print(df['is_hogwarts'].value_counts())
print()
print(df['hogwarts_house'].value_counts())

## 💼 Section 6: Engineer `profession_group` from `Profession`

Collapse the many overlapping fine-grained labels (Quidditch position variants, ministry roles,
etc.) into a handful of broad buckets. `Unknown` stays as its own class.

In [ ]:
def group_profession(val):
    s = str(val).strip()
    if s.lower() == 'unknown':
        return 'Unknown'
    s_low = s.lower()
    if 'quidditch' in s_low:
        return 'Quidditch'
    if 'minister' in s_low or 'auror' in s_low or 'ministry' in s_low:
        return 'Ministry'
    if 'author' in s_low or 'inventor' in s_low or 'professor' in s_low or 'teacher' in s_low:
        return 'Academic/Author'
    if 'healer' in s_low:
        return 'Healer'
    if 'wandmaker' in s_low or 'potioneer' in s_low or 'shopkeeper' in s_low or 'merchant' in s_low:
        return 'Magical Trade'
    return 'Other'

df['profession_group'] = df['Profession'].apply(group_profession)
df['profession_group'].value_counts()

## 📝 Section 7: Engineer `has_description` from `Descr`, drop raw text

The free-text bio isn't usable as a raw model feature. Keep only whether one exists.

In [ ]:
df['has_description'] = df['Descr'].notna().astype(int)
df['has_description'].value_counts()

## 🧹 Section 8: Drop raw / unusable columns

`Link`, `Descr`, `School`, `Profession` are dropped — replaced by their engineered versions.
`Name` is kept for display only (never a model feature).

In [ ]:
df_clean = df.drop(columns=['Link', 'Descr', 'School', 'Profession'])
df_clean.head()

## ✅ Section 9: Final check & save

In [ ]:
print('=== Final clean dataset ===')
print(f'Shape: {df_clean.shape}')
print()
print('Dtypes:')
print(df_clean.dtypes)
print()
print('Remaining missing values:')
print(df_clean.isnull().sum())

In [ ]:
df_clean.to_csv('../data/hp_characters_clean.csv', index=False)
print('✅ Saved to hp_characters_clean.csv')

---
## 🔜 What's Next?

Clean data is ready. Next: **`HP_ML_Pipeline.ipynb`** — same shared-factory pattern as LOTR:
stratified 80/20 split, `class_weight='balanced'`, logreg vs random forest per target, model
picked by macro-F1, saved to `models/hp_<target>.pkl` + `models/hp_manifest.json`.

Candidate targets: `Gender` (strong), `Species/Race` (solid), `Blood` (weak, kept for
completeness), `hogwarts_house` (moderate — likely worth restricting to `is_hogwarts==1` rows
only, otherwise `None` will dominate), `profession_group` (weak-moderate, `Unknown` still large).

> **Tell your assistant:** *"HP cleaning looks good, let's move to HP pipeline"* — and say
> whether you want `hogwarts_house` trained on the full dataset or restricted to Hogwarts-only
> rows.
